In [1]:
import warnings
from typing import override, Any

import numpy
import numpy as np
from dnnlpy.models.mlp import Module
from numpy import complex128, float64, floating

print("Numpy version:", numpy.__version__)

"""
    1.用softmax把类别分数转化为概率分布
    2.使用cross entropy 衡量预测概率和真实数据之间的差别

    在pytorch 这两步是合在一起的
"""



Numpy version: 2.5.2


'\n    1.用softmax把类别分数转化为概率分布\n    2.使用cross entropy 衡量预测概率和真实数据之间的差别\n'

In [3]:
"""
Logits: 分类模型的原始输出
"""

logits = np.array([[1.2, -0.3, 2.1, 0.7]])

In [5]:
"""
softmax: 把logits 转化成概率分布
"""


def softmax_v1(logits: np.ndarray) -> np.ndarray:
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


probs = softmax_v1(logits)
print("Predicted probabilities:", probs)
print('Sum of  probabilities:', numpy.sum(1))


def softmax_v2(logits: np.ndarray) -> np.ndarray:
    shift_logit = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shift_logit)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


probs = softmax_v1(logits)
print("Predicted probabilities:", probs)
print('Sum of  probabilities:', numpy.sum(1))

Predicted probabilities: [[0.23314023 0.05202062 0.57343245 0.1414067 ]]
Sum of  probabilities: 1
Predicted probabilities: [[0.23314023 0.05202062 0.57343245 0.1414067 ]]
Sum of  probabilities: 1


In [6]:
"""
Cross Entropy:让正确类别概率尽量大
"""


def cross_entropy(
        probs: np.ndarray, targets: np.ndarray, eps: float = 1e-12
) -> float64 | complex128 | floating[Any] | Any:
    batch_size = probs.shape[0]
    correct_probs = probs[np.arange(batch_size), targets]
    return np.mean(np.log(correct_probs + eps))


logits = np.array(
    [
        [1.2, -0.3, 2.1, 0.7],
        [-0.5, 2.3, 0.1, 1.0]
    ]
)

targets = np.array([2, 1])

probs = softmax_v2(logits)
loss = cross_entropy(probs, targets)
print("Predicted probabilities:", probs)
print('Cross Entropy  loss  :', loss)




Predicted probabilities: [[0.23314023 0.05202062 0.57343245 0.1414067 ]
 [0.042108   0.69245124 0.07672578 0.18871498]]
Cross Entropy  loss  : -0.4618163005076723


In [7]:
"""
NumPy 实现softmax Cross Entropy
"""


class CrossEntropyLoss(Module):
    def __init__(self, eps: float = 1e-12):
        super().__init__()
        self.eps = eps

    @override
    def forward(self, logits: np.ndarray, targets: np.ndarray) -> np.floating:
        probs = softmax_v2(logits)
        self.ctx = (probs, targets)
        loss = cross_entropy(probs, targets, self.eps)
        return loss

    @override
    def backward(self) -> np.ndarray:
        assert self.ctx is not None, "Must call forward before backward."
        probs, targets = self.ctx

        batch_size = probs.shape[0]
        grad = probs.copy()
        grad[np.arange(batch_size), targets] -= 1
        return grad / batch_size


loss_fn = CrossEntropyLoss()

logits = np.array(
    [
        [1.2, -0.3, 2.1, 0.7],
        [-0.5, 2.3, 0.1, 1.0]
    ]
)

targets = np.array([2, 1])

loss = loss_fn(logits, targets)
dlogits = loss_fn.backward()

print("Loss:", loss)
print("Gradient shape:", dlogits.shape)
print("Gradient:\n", dlogits)





Loss: -0.4618163005076723
Gradient shape: (2, 4)
Gradient:
 [[ 0.11657012  0.02601031 -0.21328378  0.07070335]
 [ 0.021054   -0.15377438  0.03836289  0.09435749]]
